In [1]:
import torch.nn as nn
import torch.utils.data.dataloader
from torch.nn.functional import scaled_dot_product_attention
import numpy as np
import logging
import math
from torch.optim.lr_scheduler import ExponentialLR
from src.language_models.dictionary_corpus import Corpus, Dictionary
from src.language_models.utils import batchify, load_model, get_batch, repackage_hidden
import torch.optim as optim

In [2]:
device = torch.device('cpu')
# # Old way to load data (from colorlessgreenRNNs)
corpus = Corpus('/scratch2/mrenaudin/colorlessgreenRNNs/english_data')
ntokens = len(corpus.dictionary)

eval_batch_size = 10

# Use regular batchify for all data
train_data = batchify(corpus.train, 512, device)
val_data = batchify(corpus.valid, eval_batch_size, device)
test_data = batchify(corpus.test, eval_batch_size, device)


                                  
criterion = nn.CrossEntropyLoss()

In [3]:
class SLSTM_Softmax (nn.Module):
    
    def __init__(self, vocab_size, embsz, hidden_dim, device,n_layers=2, memory_size=104, memory_dim = 5):
        super(SLSTM_Softmax, self).__init__()
        self.vocab_size = vocab_size
        self.n_layers = n_layers
        self.hidden_dim = hidden_dim
        
        self.memory_size = memory_size
        self.memory_dim = memory_dim
        
        self.lstm = nn.LSTM(embsz, self.hidden_dim, self.n_layers)

        self.W_y = nn.Linear(self.hidden_dim, vocab_size)
        self.W_n = nn.Linear(self.hidden_dim, self.memory_dim)
        self.W_a = nn.Linear(self.hidden_dim, 2)
        self.W_sh = nn.Linear (self.memory_dim, self.hidden_dim)

        self.E    = nn.Embedding(vocab_size,embsz)
        # Actions -- push : 0 and pop: 1
        self.softmax = nn.Softmax(dim=1) 
        self.sigmoid = nn.Sigmoid ()
        self.device  = device
        

        

        
    def init_hidden (self, bsz):
        return (torch.zeros (self.n_layers, bsz, self.hidden_dim).to(self.device),
                torch.zeros (self.n_layers, bsz, self.hidden_dim).to(self.device))
    def init_stack(self, bsz):
        return torch.zeros (bsz, self.memory_size,self.memory_dim).to(self.device)
    
    def forward(self, input, hidden0, stack, temperature=1.):
        """
        Forward pass of the LSTM with stack updates after each word
        
        Args:
            input: Input tensor of shape [seq_len, batch_size]
            hidden0: Initial hidden state (h0, c0) where each has shape [n_layers, batch_size, hidden_dim]
            stack: Initial stack of shape [batch_size, memory_length, memory_dim]
            temperature: Temperature for softmax (default=1.0)
            
        Returns:
            output: Output logits of shape [seq_len, batch_size, vocab_size]
            hidden: Final hidden state (h, c)
            stack: Final stack state [batch_size, memory_length, memory_dim]
        """
        seqlen, batch_size = input.size()
        
        h0, c0 = hidden0
        h0 = h0.clone()
        
        # Get stack top and transform it to match hidden state dimensions
        # stack has shape [batch_size, memory_length, memory_dim]
        stack_top = stack[:, 0, :]  # [batch_size, memory_dim]
        transformed_stack_top = self.W_sh(stack_top)  # [batch_size, hidden_dim]
        
        # Add to hidden state - using broadcasting to apply to all layers
        expanded_stack_top = transformed_stack_top.unsqueeze(0).repeat(self.n_layers, 1, 1).contiguous()
        hidden0_modified = h0 + expanded_stack_top  # [n_layers, batch_size, hidden_dim]
        
        # Embed the input sequence
        embedded = self.E(input)  # [seq_len, batch_size, embsz]
        
        # Process each word in the sequence and update the stack after each word
        current_h = hidden0_modified
        current_c = c0
        hidden = (current_h, current_c)
        
        # We need to store outputs for each timestep
        outputs = []
        
        for i in range(seqlen):
            # Process current word through LSTM
            # embedded[i] has shape [batch_size, embsz]
            # We reshape it for processing through LSTM
            current_word = embedded[i].unsqueeze(0)  # [1, batch_size, embsz]
            
            # Process through LSTM
            # ht will have shape [1, batch_size, hidden_dim]
            ht, hidden = self.lstm(current_word, hidden)
            
            # Squeeze out the singleton dimension
            ht = ht.squeeze(0)  # [batch_size, hidden_dim]
            
            # Get output for vocabulary prediction
            current_output = self.W_y(ht)  # [batch_size, vocab_size]
            outputs.append(current_output.unsqueeze(0))  # Add to our list of outputs
            
            # Compute stack action probabilities
            action_logits = self.W_a(ht)  # [batch_size, 2]
            action_weights = self.softmax(action_logits / temperature)  # [batch_size, 2]
            
            # Compute new element to potentially push to stack
            new_elt = self.sigmoid(self.W_n(ht))  # [batch_size, memory_dim]
            new_elt_expanded = new_elt.unsqueeze(1)  # [batch_size, 1, memory_dim]
            
            # Prepare push operation - move everything down and put new element at top
            push_side = torch.cat([new_elt_expanded, stack[:, :-1, :]], dim=1)
            
            # Prepare pop operation - move everything up and add a zero at the bottom
            # This is creating a tensor of zeros to fill the bottom of the stack after popping
            zeros = torch.zeros(batch_size, 1, self.memory_dim, device=self.device)
            pop_side = torch.cat([stack[:, 1:, :], zeros], dim=1)
            
            # Apply weighted action - weight of each action determines how much we push vs pop
            push_weight = action_weights[:, 0].reshape(batch_size, 1, 1)
            pop_weight = action_weights[:, 1].reshape(batch_size, 1, 1)
            
            # Update the stack as weighted combination of push and pop operations
            stack = push_weight * push_side + pop_weight * pop_side
            
            # Update hidden state for next iteration with new stack information
            if i < seqlen - 1:  # Only update if not the last timestep
                stack_top = stack[:, 0, :]  # Get new stack top
                transformed_stack_top = self.W_sh(stack_top)  # Transform to hidden dimension
                
                # Instead of just updating hidden0_modified, we need to update the actual hidden state
                # We need to add the stack information to all layers of the hidden state
                expanded_stack_top = transformed_stack_top.unsqueeze(0).repeat(self.n_layers, 1, 1)
                current_h = hidden[0] + expanded_stack_top
                current_c = hidden[1]  # Keep cell state the same
                hidden = (current_h, current_c)
        
        # Concatenate all outputs from the sequence
        output = torch.cat(outputs, dim=0)  # [seq_len, batch_size, vocab_size]
        
        return output, hidden, stack

In [4]:
model = SLSTM_Softmax(ntokens, 650, 650,device)

In [5]:
lr=0.001
optimizer = optim.Adam(model.parameters(), lr=lr)

In [6]:
train_data = batchify(corpus.train, 512, device)

In [ ]:
torch.autograd.set_detect_anomaly(True)
# Turn on training mode which enables dropout
model.train()
total_loss = 0
hidden = model.init_hidden(512)
print('init hidden', hidden[0].shape)
stack = model.init_stack(512)
print('init stack', stack.shape)
for batch, i in enumerate(range(0, train_data.size(0) - 1, 35)):
    # Get batch
    data, targets = get_batch(train_data, i, 35)
    data, targets = data.to(device), targets.to(device)
    optimizer.zero_grad()
    
    hidden = repackage_hidden(hidden)
    stack = stack.detach()
    output, hidden, stack = model(data, hidden, stack)
    print('ok')
    # Reshape outputs and targets
    output_flat = output.reshape(-1, output.size(-1))
    targets_flat = targets.reshape(-1)
    
    # Calculate loss
    loss = criterion(output_flat, targets_flat)
        
    loss.backward(retain_graph=True)

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

    optimizer.step() 


    total_loss += loss.item()

init hidden torch.Size([2, 512, 650])
init stack torch.Size([512, 104, 5])
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
ok
